In [6]:
print("=== Predicted seats (wb_wide) ===")
print(wb_wide["winner"].value_counts())

print("\n=== Actual seats (grouped) ===")
grouped["actual_winner"] = grouped[["AITC+_vote_share", "BJP_vote_share", "Other_vote_share"]].idxmax(axis=1)
grouped["actual_winner"] = grouped["actual_winner"].map({
    "AITC+_vote_share": "AITC",
    "BJP_vote_share":   "NDA",
    "Other_vote_share": "OTHER"
})
print(grouped["actual_winner"].value_counts())

print("\n=== Confusion matrix (rows=predicted, cols=actual) ===")
print(pd.crosstab(merged["winner"], merged["actual_winner"], margins=True))


=== Predicted seats (wb_wide) ===
winner
AITC    179
NDA     115
Name: count, dtype: int64

=== Actual seats (grouped) ===
actual_winner
NDA      206
AITC      78
OTHER     10
Name: count, dtype: int64

=== Confusion matrix (rows=predicted, cols=actual) ===
actual_winner  AITC  NDA  OTHER  All
winner                              
AITC             76   93     10  179
NDA               2  113      0  115
All              78  206     10  294


In [7]:
# === Seat prediction ===
merged["correct"] = merged["winner"] == merged["actual_winner"]

print("=== Overall seat accuracy ===")
print(f"Correct: {merged['correct'].sum()}  Wrong: {(~merged['correct']).sum()}  Total: {len(merged)}")
print(f"Accuracy: {merged['correct'].mean():.1%}")

print("\n=== Per-party: how many of each party's actual seats did the model call correctly ===")
print(pd.crosstab(merged["actual_winner"], merged["winner"], margins=True)
      .rename_axis("actual").rename_axis("predicted", axis=1))

print("\n=== Vote share CI coverage ===")
for party, actual, lo, hi in [
    ("AITC",  "AITC+_vote_share", "ci_low_AITC",  "ci_high_AITC"),
    ("NDA",   "BJP_vote_share",   "ci_low_NDA",   "ci_high_NDA"),
    ("OTHER", "Other_vote_share", "ci_low_OTHER", "ci_high_OTHER"),
]:
    n = len(merged)
    inside = (merged[actual] >= merged[lo]) & (merged[actual] <= merged[hi])
    print(f"{party}: {inside.sum()}/{n} constituencies ({inside.mean():.1%}) fell inside the CI")
    print(f"       CI width avg: {(merged[hi] - merged[lo]).mean():.2f}pp  |  avg miss when outside: {(merged[actual] - merged[hi]).clip(lower=0).mean() + (merged[lo] - merged[actual]).clip(lower=0).mean():.2f}pp")


=== Overall seat accuracy ===
Correct: 189  Wrong: 105  Total: 294
Accuracy: 64.3%

=== Per-party: how many of each party's actual seats did the model call correctly ===
predicted  AITC  NDA  All
actual                   
AITC         76    2   78
NDA          93  113  206
OTHER        10    0   10
All         179  115  294

=== Vote share CI coverage ===
AITC: 10/294 constituencies (3.4%) fell inside the CI
       CI width avg: 1.06pp  |  avg miss when outside: 6.83pp
NDA: 4/294 constituencies (1.4%) fell inside the CI
       CI width avg: 0.92pp  |  avg miss when outside: 13.52pp
OTHER: 8/294 constituencies (2.7%) fell inside the CI
       CI width avg: 0.60pp  |  avg miss when outside: 5.24pp
